# ChipWhisperer 응용 연구 노트북 — Husky 와이어태핑을 통한 오류주입 공격 동시 관측

## 능동(active) Lite + 수동(passive) Husky — 결합 위협 모델 실험

---

### 🎯 노트북의 목표

본 노트북은 본 연구 그룹의 두 선행 자료를 결합·확장한 **응용편** 입니다.

| 선행 자료 | 시나리오 | 본 노트북에서의 역할 |
|:----:|:----|:----|
| `FA_main.ipynb` | 단일 장치(Husky)로 클럭 글리치를 주입해 오류주입 공격 수행 | **글리치 파라미터 탐색 절차** 를 그대로 가져옴 |
| `Wiretapping4SCA.ipynb` | Lite(능동 통신) + Husky(수동 측정) 의 다중 장치 부채널 측정 | **다중 장치 동시 운용·동기화 패턴** 을 그대로 가져옴 |

두 자료를 결합한 본 노트북에서는 다음의 **결합 위협 모델(combined threat model)** 을 실험합니다.

```
┌──────────────────────────────────────────────────────────────────────────┐
│  결합 시나리오                                                              │
│                                                                          │
│   ChipWhisperer-Lite   ───  "능동 공격자" (active attacker)                 │
│      └─ 타겟 보드와 UART (SimpleSerial2) 로 통신                              │
│      └─ 타겟 펌웨어를 컴파일·플래싱                                            │
│      └─ 타겟의 시스템 클럭 공급 (HS2)                                          │
│      └─ ★ NEW: HS2 로 나가는 클럭에 **클럭 글리치** 를 합성해 주입               │
│                                                                          │
│   ChipWhisperer-Husky  ───  "은밀한 관측자" (covert observer)                │
│      └─ 트리거 / 클럭 / 전력 라인 세 가닥만 분기 측정 (wire-tap)                  │
│      └─ 통신·연산·글리치 제어에는 일체 개입하지 않음                              │
│      └─ ★ KEY INSIGHT: 클럭 라인 와이어태핑으로 **글리치 펄스 자체를 직접 관측**     │
└──────────────────────────────────────────────────────────────────────────┘
```

### 왜 이 결합 시나리오가 의미 있는가?

기존의 능동 공격(FIA)과 수동 공격(SCA)은 학술적으로 분리되어 다뤄졌지만, **실제 위협 모델** 에서는 한 명의 공격자가 두 능력을 동시에 보유하는 경우가 일반적입니다.

| 결합으로 얻는 연구적 가치 | 설명 |
|:----|:----|
| **글리치 파형의 외부 검증** | Lite가 *주입하려는* 글리치 vs. Husky가 *실제로 측정한* 글리치 파형 비교 |
| **오류 발생 시점의 물리적 단서** | 결과 코드별(loop-skip / 1-byte fault 등) 전력 파형을 비교해 **글리치가 어느 인스트럭션에 작용했는지** 추정 |
| **DFA와 SCA의 융합 가능성 검토** | 동일 시행에서 *오류가 발생한 출력* 과 *그때의 전력 파형* 을 동시 확보 → SIFA(Statistical Ineffective Fault Attack) 등 차세대 공격으로 확장 가능 |
| **실험 재현성 향상** | 글리치가 "실패" 한 경우에도 Husky 파형으로 원인 (글리치가 실제로 안 들어감 / 들어갔지만 효과 없음) 구분 |

### 노트북 구조

| 단계 | 내용 | 핵심 산출물 |
|:----:|:----|:----|
| **1단계** | 다중 장치(Lite + Husky) 동시 연결 | `lite_scope`, `husky_scope` |
| **2단계** | Lite ↔ 타겟 UART 채널 (SimpleSerial2) 확보 | `target` 객체 |
| **3단계** | Lite 경유로 타겟 펌웨어 빌드·플래싱 | 플래싱 완료된 타겟 |
| **4단계** | Golden Model 비교로 통신 검증 | 검증 완료 |
| **5단계** | **Lite 측 글리치 모듈 설정 + FIA용 1 sample = 1 clock 정렬** | `cglitch_setup()` 완료된 Lite |
| **6단계** | **Husky 측 와이어태핑 환경 (외부 클럭 + 트리거 + ADC) 구성** | 측정 준비 완료된 Husky |
| **7단계** | 베이스라인 (글리치 없음) 동시 캡처 + 글리치 시점 후보 식별 | `expected_ret`, 베이스라인 파형 |
| **8단계** | 글리치 파라미터 탐색 + Husky 와이어태핑 동시 측정 | 결과 6단계 분류 + 코드별 Husky 파형 |
| **9단계** | 통계 분석으로 최적 글리치 파라미터 도출 | `(i_offset, i_width)` 최빈값 |
| **10단계** | Bokeh 시각화 — 파라미터 분포 + **결과 코드별 와이어태핑 파형 비교** | 인터랙티브 그래프 |

> 본 자료는 FA_main / Wiretapping4SCA 가 이미 다룬 기본기 (SimpleSerial, `my_fsr_cmd` 헬퍼, 6단계 결과 분류, Bokeh 사용법 등) 의 반복 설명을 최소화하고, **두 노트북의 결합 지점** 에 집중합니다.


---

## 🔍 시작하기 전에

### 단일 장치 FIA vs. 본 노트북의 결합 FIA

| 항목 | 단일 장치 FIA (`FA_main`) | 결합 FIA (본 노트북) |
|:----:|:----:|:----:|
| 글리치 발생 주체     | 단일 scope (Husky)        | **Lite** (Husky는 무관여) |
| 통신 / 프로그래밍    | 단일 scope                | **Lite** (전담) |
| 측정 장치           | 단일 scope (자기 자신 측정)  | **Husky** (외부 와이어태핑) |
| `phase_shift_steps` | ~6000 (Husky 기준)        | **Lite 기준 (소수)** → 셀에서 자동 조회·정규화 |
| 글리치 파형 검증     | 외부 검증 수단 없음          | **Husky 클럭 라인 와이어태핑으로 직접 확인** |
| 오류 시점 분석       | `trig_count` 만으로 추정    | **Husky 전력 파형의 결과 코드별 패턴** 으로 보강 |

> ⚠️ **`phase_shift_steps` 차이의 의미**
> Husky는 한 클럭 주기를 ~6000 단계로 위상 분해하지만, Lite는 보통 그보다 훨씬 적은 단계 수를 갖습니다.
> 따라서 `FA_main` 의 `i_offset = [10, 20, 30]`, `i_width = [2090, 2100, 2110]` 같은 절대값을 그대로 옮기면 Lite 에서는 **유효 범위를 벗어날 수 있습니다.**
> 본 노트북은 `lite_scope.glitch.phase_shift_steps` 를 런타임에 조회해 **상대비율(percent of clock period)** 기준으로 탐색 범위를 정의합니다.


### 실험 환경 (물리적 배선)

```
호스트 PC (Jupyter)
    │
    │  USB ×2
    ├───────────────────────────┬────────────────────────────┐
    ▼                                                        ▼
┌──────────────────────────┐                  ┌──────────────────────────┐
│   ChipWhisperer-Lite     │                  │  ChipWhisperer-Husky      │
│   active comm + glitcher │                  │  passive wire-tap         │
└──────────────────────────┘                  └──────────────────────────┘
    │ 20-pin 커넥터                                 │ 전면 20-pin + 측면 SMA
    │                                              │
    │  HS2  ── CLKIN  (시스템 클럭 + ★글리치★ 합성)     │  D0       ← CW308 TRIG  (TRIG/GPIO4)
    │  TX   ── RX                                  │  AUX MCX  ← CW308 CLKIN  (★ 글리치 합성된 클럭 ★)
    │  RX   ── TX                                  │  Measure(Pos) ← CW308 SHUNTL
    │  IO4  ← TRIG  (Lite의 자체 트리거)             │
    │  ...                                         │
    ▼                                              ▼
        ┌────────────────────────────────────────────┐
        │   CW308 UFO 보드 + STM32F303 (타겟 MCU)      │
        │                                            │
        │   ※ TRIG (GPIO4) 신호를 Lite IO4 와         │
        │      Husky D0 양쪽에 **분기 (Y-cable)** 해야 │
        │      두 장비가 동일 시점에 정렬됩니다.          │
        └────────────────────────────────────────────┘
```

**연결 핵심 4선 — Wiretapping4SCA 대비 추가/변경 사항**

| 라인 | 출처 (타겟) | 입력 (Husky) | 비고 |
|:----:|:----:|:----:|:----|
| 트리거 | GPIO4 / TRIG | 전면 20-pin D0 | **★ Y-분기 필요** (Lite IO4 와 동시 입력) |
| 클럭   | CLKIN         | 전면 AUX MCX   | **★ 글리치된 클럭** 을 외부에서 관측 |
| 전압   | SHUNTL        | 측면 Measure (Pos) | 션트 양단 차동 입력 (전력) |
| (Lite 측) | HS2 → CLKIN | — | Lite가 클럭+글리치를 공급하는 라인 |

> 💡 **글리치된 클럭이 보이는 위치**
> Lite의 `cglitch_setup()` 은 `clock_xor` 모드에서 클럭과 글리치 펄스를 XOR 합성해 HS2 핀으로 내보냅니다.
> 이 HS2 출력이 CW308 CLKIN 에 직접 연결되므로, **CLKIN 노드를 Husky 의 AUX MCX 로 분기 측정하면 글리치 펄스가 합성된 클럭 파형을 그대로 관찰** 할 수 있습니다.
> 이는 단일 Husky 구성에서는 *간접 추정만 가능했던 글리치 형태* 를 **직접 검증** 할 수 있게 해주는 핵심 이점입니다.

### 📖 본 노트북 전용 추가 용어집

| 용어 | 의미 |
|:----|:----|
| **결합 위협 모델 (combined threat model)** | 한 공격자가 능동(injection) 능력과 수동(observation) 능력을 동시에 보유한 시나리오 |
| **글리치 파형의 외부 검증** | 능동 장치(Lite) 가 의도한 글리치를 외부 측정 장치(Husky) 가 실제로 관측해 검증하는 절차 |
| **오류 신호 (fault signature)** | 오류 결과(loop-skip, 1-byte fault 등) 가 발생할 때 전력 파형에 나타나는 특징적 패턴 |
| **트리거 분기 (trigger fan-out)** | 단일 트리거 신호를 두 측정 장치에 동시 인가하기 위한 물리적 Y-분기 |

---

# 📦 1단계 — 라이브러리 임포트 및 다중 장치 연결

> **이 단계의 목표**
> ChipWhisperer Python API와 보조 헬퍼를 로드하고, 호스트 PC 에 연결된 **Lite + Husky 두 장치를 시리얼 넘버 기반으로** 동시에 객체화합니다.
> 이 절차는 `Wiretapping4SCA.ipynb` 의 1단계와 동일합니다 — FIA 전용 변경 사항은 없습니다.

---

### 1.1 헬퍼 로드 및 상수 정의

`My_script.ipynb` 는 `my_fsr_cmd()`, Bokeh 임포트, 시각화 함수(`plot_t`), 시드 고정 등을 일괄 로드합니다.

In [ ]:
# 사전 정의된 헬퍼 (my_fsr_cmd, plot_t 등) 로드
%run My_script.ipynb

import chipwhisperer as cw
import logging
import time
import numpy as np
import pandas as pd
import scipy as sp
from tqdm.notebook import trange, tqdm

# ─────────────────────────────────────────
# 타겟 / 펌웨어 관련 상수
# ─────────────────────────────────────────
PLATFORM      = 'CW308_STM32F3'   # 타겟 보드 종류
SCOPETYPE     = 'OPENADC'         # 캡처 장치 (Lite/Husky 공통)
CRYPTO_TARGET = 'NONE'            # 자체 펌웨어 (XOR 루프) 사용
SS_VER        = 'SS_VER_2_1'      # SimpleSerial 프로토콜 버전

### 1.2 두 장치 동시 검출 및 객체 분리

`cw.list_devices()` 가 반환하는 시리얼 넘버를 명시해 `cw.scope(sn=...)` 로 연결합니다.
시리얼 넘버 없이 `cw.scope()` 만 호출하면 USB 버스에서 가장 먼저 발견된 장치 하나만 잡히므로 다중 장치 구성에서는 반드시 명시 연결이 필요합니다.

> 💡 이 셀은 `Wiretapping4SCA.ipynb` 의 동명 셀과 **완전히 동일** 합니다. 두 장치의 역할은 5/6단계에서 분기됩니다.

In [ ]:
def connect_all_devices() -> dict:
    """연결된 모든 ChipWhisperer 장치에 접속하여 딕셔너리로 반환"""
    device_list = cw.list_devices()

    if not device_list:
        raise RuntimeError("연결된 ChipWhisperer 장치가 없습니다.")

    print(f"발견된 장치 수: {len(device_list)}\n")
    scopes = {}

    for device in device_list:
        # 'ChipWhisperer-Lite' → 'ChipWhisperer_Lite' 로 변환해 dict 키로 사용
        name = device['name'].replace("-", "_")
        sn   = device['sn']

        try:
            scopes[name] = cw.scope(sn=sn)
            print(f"  [✓] {name} 연결 완료  (SN: {sn})")
        except Exception as e:
            print(f"  [✗] {name} 연결 실패  (SN: {sn})\n      └─ {e}")

    return scopes

# PC에 연결된 모든 장치 인식 및 할당
scopes = connect_all_devices()
lite_scope  = scopes["ChipWhisperer_Lite"]
husky_scope = scopes["ChipWhisperer_Husky"]

---

# 🔌 2단계 — Lite 를 통한 타겟 보드 통신 채널 확보

> **이 단계의 목표**
> 타겟 보드(STM32F303)와의 **모든 시리얼 통신은 Lite 가 전담** 합니다.
> SimpleSerial2 객체를 `lite_scope` 위에 바인딩해 `target` 객체를 생성합니다.

---

이후 등장하는 모든 `target.send_cmd()`, `target.read_cmd()`, `my_fsr_cmd(target, ...)` 호출은 **Lite 의 UART 핀** 을 경유합니다.
Husky 는 이 통신에 일체 개입하지 않으며, 8단계 와이어태핑 시 외부에서 트리거·클럭·전력 라인만 관측합니다.

> ⚠️ **`target` 객체는 `lite_scope` 에만 묶인다는 점**
> 모든 명령은 Lite 의 UART 를 통해 전송됩니다. 만약 이후 단계에서 `cw.target(husky_scope, ...)` 를 호출하면 채널이 Husky 로 바뀌어 본 시나리오의 의도와 어긋납니다 — **호출 금지.**

In [ ]:
# Lite를 통한 타겟 보드 연결 설정
if SS_VER == "SS_VER_2_1":
    target_type = cw.targets.SimpleSerial2
else:
    raise OSError("지원되지 않는 SimpleSerial 버전입니다.")

try:
    target = cw.target(lite_scope, target_type)
    print("\n[✓] ChipWhisperer_Lite 에 타겟 보드 연결 성공")
except Exception as e:
    print(f"\n[✗] ChipWhisperer_Lite 에 타겟 보드 연결 실패: {e}")

---

# 🛠 3단계 — 펌웨어 빌드 및 Lite 경유 타겟 프로그래밍

> **이 단계의 목표**
> `simpleserial_main/` 디렉터리의 펌웨어를 STM32F303 용으로 컴파일하고, **Lite 의 SWD 인터페이스** 로 타겟에 플래싱합니다.
> 이는 `Wiretapping4SCA.ipynb` 의 3단계와 동일합니다.

---

| 하위 단계 | 동작 | 비고 |
|:----:|:----|:----|
| ① 컴파일       | `make PLATFORM=... CRYPTO_TARGET=NONE SS_VER=SS_VER_2_1` | `subprocess.run` |
| ② 프로그래머 선택 | `cw.programmers.STM32FProgrammer` | STM 계열용 |
| ③ Lite 기본 셋업 | `lite_scope.default_setup()` | Lite 의 클럭·UART·HS2 정상화 |
| ④ 플래싱       | `cw.program_target(lite_scope, prog, hex_path)` | **Lite 가 프로그래머** |
| ⑤ 빌드 산출물 정리 | `make clean` | 작업 디렉터리 청결 |

> 💡 **`lite_scope.default_setup()` 의 부수 효과**
> 이 호출은 Lite 의 게인·ADC·트리거 모드를 표준값으로, **HS2 를 타겟 클럭 공급원** 으로 설정합니다.
> 본 노트북에서는 5단계의 `cglitch_setup()` 호출이 HS2 의 출력 모드를 *"클럭 + 글리치 XOR"* 로 변경하므로, 그 시점부터 HS2 가 글리치를 합성해 내보내기 시작합니다.

In [ ]:
# 1. 펌웨어 컴파일
print("펌웨어 컴파일 중...")
compile_cmd = ["make", f"PLATFORM={PLATFORM}", f"CRYPTO_TARGET={CRYPTO_TARGET}", f"SS_VER={SS_VER}"]
subprocess.run(compile_cmd, cwd="simpleserial_main/", capture_output=True)
print("펌웨어 컴파일 완료")

# 2. 프로그래머 선택
if "STM" in PLATFORM or PLATFORM == "CWLITEARM" or PLATFORM == "CWNANO":
    prog = cw.programmers.STM32FProgrammer
else:
    raise OSError("프로그래머가 지원되지 않는 플랫폼입니다.")

# 3. 펌웨어 플래싱 (Lite 가 프로그래머 역할)
lite_scope.default_setup()
try:
    hex_path = f"simpleserial_main/simpleserial-base-{PLATFORM}.hex"
    cw.program_target(lite_scope, prog, hex_path)
    print(f"[✓] {PLATFORM} 타겟 보드에 프로그램 업로드 완료")
except Exception as e:
    print(f"[✗] 펌웨어 프로그램 실패: {e}")

# 4. 빌드 산출물 정리
print("펌웨어 컴파일 클린 중...")
compile_cmd = ["make", f"PLATFORM={PLATFORM}", f"CRYPTO_TARGET={CRYPTO_TARGET}", f"SS_VER={SS_VER}", "clean"]
subprocess.run(compile_cmd, cwd="simpleserial_main/", capture_output=True)
print("펌웨어 컴파일 클린 완료")

---

# ✅ 4단계 — Golden Model 통신 검증

> **이 단계의 목표**
> 글리치를 *주입하지 않은* 상태에서 Lite ↔ 타겟 통신·연산이 정상인지를 Golden Model 로 확인합니다.
> 본 검증 통과 이후의 단계에서 보이는 비정상 결과는 모두 **글리치의 영향** 으로 해석할 수 있습니다.

---

타겟 펌웨어는 다음과 같은 단순 XOR 연산을 수행합니다:

```c
for (i = 0; i < global_len; i++) {
    output[i] = key[i] ^ plaintext[i];
}
```

호스트에서 동일한 `k ⊕ p` 를 직접 계산한 값(`Golden_k_XOR_p`) 이 타겟 반환값과 일치하면 통신·연산이 정상입니다.

> 💡 **단순 XOR 인 이유**
> 본 노트북의 목적은 *결합 시나리오 절차의 정립* 이므로, 글리치 효과를 명확히 관찰할 수 있도록 펌웨어를 단순화했습니다.
> AES 등 실제 암호 연산에 대한 DFA 도 동일 절차를 그대로 적용할 수 있습니다.

In [ ]:
MAX_DATA_LEN = 40  # 본 노트북 전반에서 사용할 출력 바이트 수 (= for-loop 반복 횟수)

# 재현성 시드 고정
random.seed(1)

# 무작위 키(k), 평문(p) 생성 + 골든 결과(k XOR p) 사전 계산
data_k = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
data_p = bytearray(random.randint(0, 255) for _ in range(MAX_DATA_LEN))
Golden_k_XOR_p = bytes(x ^ y for x, y in zip(data_k, data_p))

# 0x81 = 데이터 전송  ('k'=key, 'p'=plaintext, 'l'=length)
# 0x82 = 연산 트리거
# 0x83 = 결과 회수
my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))
my_fsr_cmd(target, 0x82, 'c', [])
Return_k_XOR_p = my_fsr_cmd(target, 0x83, 'r', [], payload_only=True)

print('=== 결과 비교 ===')
print(f'타겟 결과 : {Return_k_XOR_p.hex(" ")}')
print(f'골든 모델 : {Golden_k_XOR_p.hex(" ")}')
print()
if Golden_k_XOR_p == Return_k_XOR_p:
    print('[✓] 통신 및 연산 검증 성공! (타겟 출력 == 골든 모델)')
    expected_ret = bytes(Return_k_XOR_p)   # 이후 단계에서 글리치 효과 판정의 기준값
else:
    print('[✗] 불일치! 통신 오류 또는 펌웨어 오류를 확인하세요.')

---

# ⚡ 5단계 — Lite 측 글리치 모듈 설정 + FIA 용 1 sample = 1 clock 정렬

> **이 단계의 목표**
> Lite 의 글리치 모듈을 **클럭 글리치(XOR 합성)** 모드로 활성화하고, Lite 자체 ADC 를 *1 ADC 샘플 = 1 타겟 클럭* 으로 정렬해 글리치 시점(`ext_offset`) 과 샘플 인덱스를 1:1 로 매핑합니다.
> 본 단계는 `FA_main.ipynb` 의 1단계와 거의 동일하지만, **대상 객체가 `lite_scope` 라는 점** 이 핵심 차이입니다.

---

### 5.1 왜 1 sample = 1 clock 인가? (복습)

오류주입 분석에서는 *몇 번째 클럭에서 글리치가 발생하는지* 가 가장 중요한 정보입니다.
SCA 와 달리 미세 누설 패턴 분석이 아니라, **"이 클럭에 명령어 X 가 실행 중"** 이라는 시간축 매핑이 필요합니다.

```
ADC 샘플레이트 = 타겟 클럭 × adc_mul × decimate_inverse

  · adc_mul = 4   → 1 클럭당 4 샘플 (SCA 미세 누설 분석용)
  · adc_mul = 1   → 1 클럭당 1 샘플 (FIA 시점 식별용)  ← 본 단계 (Lite)
  · decimate = 1  → 다운샘플링 없음
```

> 🔬 **본 노트북의 두 ADC 는 서로 다른 `adc_mul` 을 사용합니다**
> - **Lite**  (5단계, FIA): `adc_mul = 1` → 클럭 단위 정밀 글리치 타이밍
> - **Husky** (6단계, 와이어태핑): `adc_mul = 4` → 클럭 *내부* 의 글리치 펄스 형태를 관찰
>
> 두 장비를 의도적으로 다른 시간 분해능으로 운용해, **글리치의 타이밍(언제) 과 모양(어떻게)** 을 양면에서 동시에 포착합니다.

### 5.2 Lite 의 `cglitch_setup()`

| 함수 | 활성 글리치 | 출력 핀 |
|:----|:----|:----|
| `lite_scope.cglitch_setup()` | **클럭** 글리치 (XOR 합성) | HS2 → CW308 CLKIN |
| `lite_scope.vglitch_setup()` | **전압** 글리치 (HP/LP 트랜지스터) | crowbar 라인 |

본 노트북은 클럭 글리치를 기준으로 진행합니다 (전압 글리치 확장은 `FA_main.ipynb` 의 주석 참조).

In [ ]:
# ── 1) Lite 글리치 모듈 활성화 ────────────────────────
lite_scope.cglitch_setup()  # clock glitch (XOR 합성) 모드

# ── 2) Lite ADC 클럭 동기화: 1 샘플 = 1 클럭 ───────────
# (Husky 는 다음 단계에서 별도로 4× 오버샘플링으로 설정)
lite_scope.clock.adc_src    = 'clkgen_x1'  # ADC 클럭 = 타겟 클럭 (×1)
lite_scope.clock.clkgen_src = 'system'     # 클럭 소스: 시스템 PLL
lite_scope.clock.adc_mul    = 1            # 오버샘플링 없음 (FIA 용)
lite_scope.adc.decimate     = 1            # 다운샘플링 없음

# ── 3) 타겟 보드 리셋 (2회 반복으로 안정화) ─────────────
def reset_target_via_lite(scope):
    """Lite 의 nRST 라인을 통해 타겟 MCU 리셋"""
    scope.io.nrst = 'low'
    time.sleep(0.05)
    scope.io.nrst = 'high_z'
    time.sleep(0.05)

reset_target_via_lite(lite_scope); time.sleep(1)
reset_target_via_lite(lite_scope); time.sleep(1)

# ── 4) 설정 검증 ──────────────────────────────────────
print('=== Lite ADC / 클럭 설정 ===')
print(f'  clock.adc_src       : {lite_scope.clock.adc_src}')
print(f'  clock.clkgen_src    : {lite_scope.clock.clkgen_src}')
print(f'  clock.adc_mul       : {lite_scope.clock.adc_mul}')
print(f'  adc.decimate        : {lite_scope.adc.decimate}')
print()
print(f'  glitch.phase_shift_steps : {lite_scope.glitch.phase_shift_steps}')
print(f'  glitch.output            : {lite_scope.glitch.output}')
print()
print('[✓] Lite: 1 ADC 샘플 = 1 타겟 클럭 정렬 완료')
print('[✓] Lite: cglitch_setup 으로 HS2 가 클럭+글리치 XOR 모드로 전환됨')

### 5.3 Lite 의 `phase_shift_steps` 확인 + 글리치 파라미터 범위 산출

Husky 기준 `~6000`, Lite 기준 그보다 적은 단계 수가 출력됩니다.
이 값을 8단계의 탐색 범위 자동 산정에 사용합니다.

In [ ]:
# 런타임 조회된 phase_shift_steps 기준 범위
PS = lite_scope.glitch.phase_shift_steps

print(f'Lite phase_shift_steps : {PS}')
print(f'  · i_offset 허용 범위 : 0 ~ {PS}')
print(f'  · i_width  허용 범위 : 0 ~ {PS // 2}')
print(f'  · 1 step 당 위상   : 약 360°/ {PS} = {360.0/PS:.4f}° (이론치)')

---

# 📡 6단계 — Husky 측 와이어태핑 환경 구성

> **이 단계의 목표**
> Husky 가 **통신·글리치 제어에는 일체 개입하지 않으면서** 트리거·클럭·전력 세 신호선을 외부에서 정확히 동기 측정하도록 구성합니다.
> 절차는 `Wiretapping4SCA.ipynb` 의 5단계와 동일하지만, 본 노트북에서는 Husky 가 관측하는 클럭 라인이 *글리치가 합성된 클럭* 이라는 점이 결정적 차이입니다.

---

### 6.1 와이어태핑의 클럭 동기화 전략 (복습)

Husky 는 클럭을 *공급* 하는 입장이 아니라 *수신* 하는 입장이므로 다음 단계로 외부 클럭에 자신을 정렬해야 합니다.

```
[1] PLL 입력 소스를 외부 AUX 로 전환
[2] AUX MCX 핀을 high-Z 입력 모드로 설정
[3] 내장 주파수 카운터로 외부 클럭 주파수 측정
[4] 측정 주파수의 최빈값으로 PLL 잠금
[5] ADC 클럭을 타겟 클럭의 4배로 정렬 (adc_mul=4)  ← Lite 와 다름
[6] ADC 리셋 후 lock 상태 확인
```

> 💡 **본 노트북의 Husky 가 보는 클럭은 *글리치된 클럭* 입니다**
> CW308 CLKIN 은 Lite 의 HS2 로부터 *클럭 + 글리치 XOR 합성* 신호를 받습니다.
> Husky 는 이 라인을 AUX MCX 로 분기 측정하므로, **PLL 잠금에는 글리치 발생 *전* 의 깨끗한 베이스라인에서 수행** 해야 안정적입니다 (즉, 본 단계는 베이스라인에서 한 번만 수행).
>
> 8단계 탐색 중 글리치가 활성화되면 외부 클럭에 일시적 위상 흔들림이 생기지만, PLL 의 hysteresis 가 lock 을 유지합니다 (jitter 가 너무 크면 lock 이 풀릴 수 있어 모니터링이 필요).

In [ ]:
# 이전 설정의 간섭을 방지하기 위해 Husky 공장 초기화
husky_scope.default_setup()
time.sleep(0.5)

print("Husky 스코프 하드웨어 동기화 및 튜닝 중...")

# ---------------------------------------------------------
# [1] 클럭 신호 탐색 및 동기화 (Aux in/out)
# ---------------------------------------------------------
husky_scope.clock.clkgen_freq = 0
husky_scope.clock.reset_adc()
# AUX MCX 를 입력(high-Z) 으로 설정 → Husky 가 클럭을 driving 하지 않고 수신만 함
husky_scope.io.aux_io_mcx = 'high_z'
# PLL 입력 소스를 외부 클럭(extclk_aux_io) 으로 지정
husky_scope.clock.clkgen_src = 'extclk_aux_io'
husky_scope.clock.reset_adc()

if (husky_scope.io.aux_io_mcx == 'high_z') and (husky_scope.clock.clkgen_src == 'extclk_aux_io'):
    print(f"[✓] io.aux_io_mcx     = {husky_scope.io.aux_io_mcx}")
    print(f"[✓] clock.clkgen_src  = {husky_scope.clock.clkgen_src}")
else:
    print(f"[✗] 외부 클럭 설정 실패")

### 6.2 외부 클럭 주파수 탐색 + ADC 4× 오버샘플링 정렬

핵심 한 줄:

```python
husky_scope.clock.clkgen_freq = husky_scope.clock.freq_ctr
```

`freq_ctr` 는 Husky 내장 주파수 카운터의 실시간 외부 클럭 측정값입니다.
20회 측정 후 최빈값(`mode`) 을 PLL 목표로 대입해 안정적인 잠금을 도모합니다.

| 설정 | 값 | 의미 |
|:----:|:----:|:----|
| `freq_ctr_src` | `'extclk'` | 카운터 측정 대상 = 외부 클럭 |
| `clkgen_freq`  | `freq_ctr` 의 mode | 측정된 주파수에 PLL 잠금 |
| `adc_mul`      | **4** | 1 클럭 → 4 ADC 샘플 (글리치 펄스 형태 관찰용) |

> 🔬 **본 노트북 고유 — `adc_mul = 4` 의 의미**
> 클럭 1주기 내에서 글리치 펄스가 차지하는 위상은 매우 짧습니다(약 30~40%).
> 4× 오버샘플링이면 *글리치가 켜진 구간* 과 *원본 클럭 구간* 이 4개의 다른 샘플 값으로 표현되어 **글리치 펄스의 폭·위상을 직접 관찰** 할 수 있습니다.
> 1× 샘플링은 글리치 발생 *여부* 만 알 수 있고 *형태* 는 알 수 없습니다.

In [ ]:
# ---------------------------------------------------------
# [2] 외부 클럭 주파수 탐색 및 동기화
# ---------------------------------------------------------
# 더 정밀한 클럭 주파수 설정 (PLL 잠금 실패 시 주석 처리)
husky_scope.clock.pll._allow_rdiv = True
# 주파수 카운터의 측정 대상을 외부 클럭으로 지정
husky_scope.clock.freq_ctr_src = 'extclk'
# 카운터 초기 안정화 대기
time.sleep(0.5)

# 주파수 데이터 수집 (0.2초 간격, 20회)
freqs = []
for _ in range(20):
    freqs.append(husky_scope.clock.freq_ctr)
    time.sleep(0.2)
data = pd.Series(freqs)
print(f"최빈값: {data.mode().iloc[0]} (등장 {(data == data.mode().iloc[0]).sum()}/{len(data)}회)")
print(f"범위:   {data.min()} ~ {data.max()} (Δ={data.max()-data.min()})")
print("\n[전체 통계 요약]")
print(data.describe())

# 외부 클럭 주파수에 PLL 잠금
husky_scope.clock.clkgen_freq = data.mode().iloc[0]
# ADC 샘플레이트 = 타겟 클럭 × 4 (4× 오버샘플링)
husky_scope.clock.adc_mul = 4
husky_scope.clock.reset_adc()

if husky_scope.clock.adc_locked:
    print("[✓] ADC 클럭 동기화 완료")
    print(f"   - ADC 샘플레이트 (adc_freq): {husky_scope.clock.adc_freq:,.0f} Hz")
else:
    print("[✗] ADC 클럭 동기화 실패 (Lock Error)")

if husky_scope.clock.clkgen_locked:
    print("[✓] Husky PLL 잠금 성공")
    print(f"   - 타겟 클럭 (clkgen_freq) : {husky_scope.clock.clkgen_freq:,.0f} Hz")
else:
    print("[✗] Husky PLL 잠금 실패! 외부 클럭의 진폭/듀티/안정성을 확인하세요.")

### 6.3 트리거 입력 핀 및 ADC 캡처 파라미터 설정

| 설정 | 값 | 의미 |
|:----:|:----:|:----|
| `trigger.triggers` | `'userio_d0'` | 전면 USERIO D0 핀을 트리거 입력 |
| `trigger.module`   | `'basic'`     | 단순 엣지 검출 |
| `adc.basic_mode`   | `'rising_edge'` | 상승 엣지에서 캡처 시작 |
| `gain.db`          | `25`          | LNA 게인 |
| `adc.samples`      | `5000`        | 한 번의 캡처에서 수집할 샘플 수 |

> 🔬 **Husky `adc.samples` 산정**
> 본 펌웨어의 XOR 루프는 약 200 클럭 내에 완료됩니다 (트리거 직후 셋업 ~30 clk + 루프 ~170 clk + 마무리 ~20 clk).
> `adc_mul = 4` 이므로 200 × 4 = 800 샘플이면 본 연산은 모두 포함되지만, 글리치 효과로 인한 추가 명령 실행, 비정상 분기 등을 여유 있게 포착하기 위해 **5000 샘플** (≈ 1250 클럭) 로 설정합니다.

In [ ]:
# 트리거 입력 = 전면 USERIO D0 (CW308 의 GPIO4 / TRIG 를 분기해 입력)
husky_scope.trigger.triggers = 'userio_d0'
husky_scope.trigger.module   = 'basic'
husky_scope.adc.basic_mode   = 'rising_edge'

# 캡처 파라미터
husky_scope.gain.db        = 25      # LNA 게인 (dB)
husky_scope.adc.samples    = 5000    # 1 캡처당 샘플 수
husky_scope.adc.offset     = 0       # 트리거 후 캡처 시작점 (0 = 즉시)
husky_scope.adc.presamples = 0       # 트리거 이전 사전 캡처 안 함

print(f"[✓] Husky 스코프 캡처 파라미터 설정 완료")
print(f"  trigger.triggers : {husky_scope.trigger.triggers}")
print(f"  trigger.module   : {husky_scope.trigger.module}")
print(f"  adc.basic_mode   : {husky_scope.adc.basic_mode}")
print(f"  gain.db          : {husky_scope.gain.db}")
print(f"  adc.samples      : {husky_scope.adc.samples}")
print(f"  adc.offset       : {husky_scope.adc.offset}")

---

# 🌊 7단계 — 베이스라인 (글리치 없음) 동시 캡처 + 글리치 시점 후보 식별

> **이 단계의 목표**
> 글리치를 **주입하지 않은 정상 상태** 의 파형을 Lite·Husky 두 스코프로 동시에 수집해
> (a) 정상 출력값 `expected_ret` 를 재확인 (4단계 값과 일치해야 함),
> (b) Lite 의 `trig_count` 로 *연산이 차지한 클럭 수* 를 측정해 글리치 시점 후보 범위를 정합니다.
> (c) Husky 의 와이어태핑 파형으로 *연산 구간을 시각적으로* 식별합니다.

---

### 7.1 글리치 비활성 모드 (`arm_timing = 'no_glitch'`)

`scope.glitch.arm_timing = 'no_glitch'` 로 두면, `arm()` 호출 후 트리거가 들어와도 **글리치 펄스가 출력되지 않습니다.**
이 모드를 사용해 *글리치 비활성 + 캡처 활성* 상태로 베이스라인을 수집합니다.

| `arm_timing` 값 | 의미 |
|:----:|:----|
| `'after_scope'` | scope arm 직후부터 글리치 활성 (8단계 탐색용) |
| `'no_glitch'`   | 글리치 비활성 (베이스라인 수집용 / 디버그용) |

In [ ]:
# Lite 측: 글리치 비활성 + clock_xor 모드 유지
lite_scope.glitch.arm_timing = 'no_glitch'
lite_scope.glitch.output     = 'clock_xor'

# 타겟 초기화 + 데이터 주입
reset_target_via_lite(lite_scope)
my_fsr_cmd(target, 0x81, 'k', data_k)
my_fsr_cmd(target, 0x81, 'p', data_p)
my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

# Lite ADC 샘플 수 임시 충분값 (첫 캡처 후 trig_count 로 재설정)
lite_scope.adc.samples = 24400

# 양쪽 스코프 모두 arm → 캡처 → 트레이스 회수
target.flush()
husky_scope.arm()
lite_scope.arm()
target.send_cmd(cmd=0x82, scmd=ord('c'), data=[])
ack = target.read_cmd(timeout=500)

ret_husky = husky_scope.capture()
ret_lite  = lite_scope.capture()
if ret_husky or ret_lite:
    raise RuntimeError(f"베이스라인 캡처 실패: husky={ret_husky}, lite={ret_lite} — 배선·트리거 점검 필요")

target.flush()
target.send_cmd(cmd=0x83, scmd=ord('r'), data=[])
ret_payload = target.read_cmd(timeout=500)

# 트레이스 회수
trace_lite_baseline  = np.array(lite_scope.get_last_trace())
trace_husky_baseline = np.array(husky_scope.get_last_trace())

# 페이로드 추출 + 검증
expected_ret_now = ret_payload[3 : 3 + ret_payload[2]]
assert expected_ret_now == expected_ret, "베이스라인 출력이 4단계 골든값과 다릅니다!"

# 다음 단계(8단계) 용: Lite samples 자동 산정 (FA_main 의 my_setting_num_samples 와 동일 원리)
trig_count_baseline = int(lite_scope.adc.trig_count)
lite_scope.adc.samples = trig_count_baseline + 50  # 마진 50 샘플

print(f'\n=== 베이스라인 ===')
print(f'  Lite  파형 길이      : {len(trace_lite_baseline)} 샘플  (≈ {len(trace_lite_baseline)} clock)')
print(f'  Husky 파형 길이      : {len(trace_husky_baseline)} 샘플  (≈ {len(trace_husky_baseline)//4} clock)')
print(f'  Lite trig_count     : {trig_count_baseline}')
print(f'  → 다음 단계용 samples : {lite_scope.adc.samples}')
print(f'  expected_ret        : {expected_ret.hex(" ")}')
print(f'  [✓] 베이스라인 동시 캡처 성공 — Husky 가 동일 트리거에 정렬 측정됨')

### 7.2 베이스라인 파형 시각화 — Husky 와이어태핑 vs Lite 자체 측정

두 파형을 **별도 그래프로** 그려 다음을 확인합니다:
- **Husky 클럭 라인 (AUX MCX)** : 정상 클럭 파형이 보이는가? (글리치 비활성이므로 깨끗한 사각파에 가까워야 함)
- **Lite 션트 측정 (1 clk/sample)** : 본 펌웨어 XOR 루프의 반복 패턴이 보이는가? (170 cycles 부근에서 주기적 특징)

> 🔬 **이 그래프에서 식별할 것**
> - 트리거 직후 셋업 구간(약 ~30 clk) 의 평탄한 영역
> - for-loop 본체 구간(약 30 ~ 200 clk)의 반복 패턴
> - 연산 종료 후 정리 구간
>
> 본 펌웨어의 경우 글리치 유효 구간은 트리거 후 **약 170 ~ 200 clk** 입니다. 이 값이 8단계 `ext_offset` 탐색 범위의 기준이 됩니다.

In [ ]:
# ── Bokeh: Lite 베이스라인 (1 sample = 1 clock) ────────
p1 = figure(
    width=900, height=280,
    title='Baseline — Lite (self-measure, 1 sample = 1 clock, no glitch)',
    x_axis_label='Sample Index (= Clock Cycle from Trigger)',
    y_axis_label='Amplitude (V)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
)
p1.line(np.arange(len(trace_lite_baseline)), trace_lite_baseline,
        line_width=1.2, line_color='#2E86AB')
p1.title.text_font_size = '12pt'
p1.grid.grid_line_alpha = 0.3
p1.outline_line_color = None
p1.add_tools(HoverTool(tooltips=[('Clock', '@x{0}'), ('V', '@y{0.0000}')], mode='vline'))

# ── Bokeh: Husky 베이스라인 (4 samples / clock — wiretap) ─
husky_x = np.arange(len(trace_husky_baseline)) / 4.0  # 샘플 → 클럭 변환
p2 = figure(
    width=900, height=280,
    title='Baseline — Husky (wire-tap on CLKIN line via AUX MCX, 4× oversample)',
    x_axis_label='Clock Cycle from Trigger  (= sample / 4)',
    y_axis_label='Amplitude (V)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
)
p2.line(husky_x, trace_husky_baseline, line_width=1.0, line_color='#A23B72', line_alpha=0.85)
p2.title.text_font_size = '12pt'
p2.grid.grid_line_alpha = 0.3
p2.outline_line_color = None
p2.add_tools(HoverTool(tooltips=[('Clock', '@x{0.00}'), ('V', '@y{0.0000}')], mode='vline'))

show(column(p1, p2))

---

# 🎯 8단계 — 글리치 파라미터 탐색 + Husky 와이어태핑 동시 측정

> **이 단계의 목표**
> `(ext_offset, offset, width)` 의 3차원 파라미터 공간을 격자(grid) 탐색하며,
> 각 시행마다 **Lite 의 자체 ADC + Husky 의 와이어태핑** 으로 동시 캡처해
> 결과 6단계 분류와 함께 **결과 코드별 와이어태핑 파형** 을 보존합니다.

---

### 8.1 8단계의 두 가지 변경점 (FA_main 대비)

| 변경점 | FA_main | 본 노트북 |
|:----:|:----:|:----:|
| arm 패턴 | `scope.arm()` 1회 | **`husky_scope.arm()` + `lite_scope.arm()`** (순서 중요) |
| 캡처 검사 | `scope.capture()` 1회 | **두 스코프 모두 `capture()` 검사** |
| 파라미터 범위 | Husky 절대값(`width=2090` 등) | **`PS` 기반 상대비율** (Lite 적응) |
| 트레이스 보존 | loop-skip 만 | **결과 코드별로 husky/lite 양쪽** |

> ⚠️ **arm 순서를 뒤바꾸지 말 것**
> `Encrypt(... )` 를 호출한 *후* 에 `arm()` 을 부르면 트리거가 이미 지나가 버려 캡처가 실패합니다.
> 두 스코프 모두 arm 된 상태에서 트리거가 한 번 발생해야 두 파형이 **시간축으로 정렬** 됩니다.

### 8.2 글리치 파라미터 범위 (Lite `PS` 기반)

`FA_main` 의 Husky 절대값은 `phase_shift_steps ≈ 6000` 을 가정한 값입니다.
Lite 의 `PS` 가 더 작을 경우 동일 절대값이 *클럭 주기보다 길어지는* 등 의미가 달라지므로, **상대비율** 로 변환합니다.

```
i_offset 후보(%): 1%, 3%, 5%, 8%, 10%  → 클럭 시작 직후의 위상들
i_width  후보(%): 30%, 35%, 40%        → loop-skip 이 잘 일어나는 폭 (경험치)
```

In [ ]:
# ── 글리치 발생 시점 + 출력 모드 설정 (탐색 활성) ─────
lite_scope.glitch.arm_timing = 'after_scope'   # scope arm 후 글리치 활성
lite_scope.glitch.output     = 'clock_xor'     # 클럭 XOR (= 클럭 글리치)

# ── 경고 로그 무시 (탐색 중 빈번하게 발생하는 정상 경고) ──
logging.getLogger('ChipWhisperer Target').setLevel(logging.ERROR)
logging.getLogger('ChipWhisperer Glitch').setLevel(logging.ERROR)

# ── 탐색 범위 (PS 기반 상대비율) ────────────────────────
PS = lite_scope.glitch.phase_shift_steps

i_ext_offsets = list(range(170, 183))   # baseline trig_count 부근. 본 펌웨어 기준 경험치
i_offsets_pct = [0.01, 0.03, 0.05, 0.08, 0.10]
i_widths_pct  = [0.30, 0.33, 0.36, 0.39]

i_offsets = sorted(set(int(PS * f) for f in i_offsets_pct))
i_widths  = sorted(set(int(PS * f) for f in i_widths_pct))

print(f'phase_shift_steps (Lite) : {PS}')
print(f'i_ext_offset 탐색 범위    : {i_ext_offsets[0]} ~ {i_ext_offsets[-1]}  ({len(i_ext_offsets)} 개)')
print(f'i_offset     후보         : {i_offsets}')
print(f'i_width      후보         : {i_widths}')
print(f'총 시행 횟수             : {len(i_ext_offsets) * len(i_offsets) * len(i_widths) * 5} (×5 반복)')

print('\n[✓] 글리치 출력 모드 활성화 완료 — arm_timing=after_scope')

### 8.3 결과 저장 자료구조

```
cglitch_result      [코드, ext_offset, width, offset]      ← 통계 분석용
cglitch_extra_data  [expected_ret, actual_ret]             ← 디버깅 검증용
husky_by_code, lite_by_code  결과 코드별 파형 (각 코드당 최대 MAX_TRACES_PER_CODE 개)
```

각 결과 코드당 보존 파형 수에 상한을 두는 이유는 메모리 보호입니다.
탐색 횟수가 수천 회에 이를 수 있는데, 각 husky 파형이 5000 샘플 × float = 약 40KB 이므로 무제한 저장 시 GB 단위로 늘어날 수 있습니다.

In [ ]:
# 결과 컨테이너
cglitch_result     = []   # [코드, ext_offset, width, offset]
cglitch_extra_data = []   # [expected_ret, actual_ret]

# 결과 코드별 파형 저장 (메모리 보호 위한 상한)
MAX_TRACES_PER_CODE = 20

husky_by_code = {0: [], 1: [], 2: [], 3: [], 4: [], 5: []}
lite_by_code  = {0: [], 1: [], 2: [], 3: [], 4: [], 5: []}

# 결과 코드별 컨텍스트 (어떤 파라미터에서 발생했는지)
ctx_by_code   = {0: [], 1: [], 2: [], 3: [], 4: [], 5: []}

print('자료구조 초기화 완료')

### 8.4 메인 탐색 루프 — Husky 와이어태핑 동시 캡처

본 셀의 흐름:

```
for each (ext_offset, offset, width):
    for each 반복 (5회):
        1. Lite 글리치 파라미터 설정
        2. 타겟 리셋 + (k, p, l) 주입
        3. ★ husky_scope.arm()  →  lite_scope.arm()   (순서 주의)
        4. 0x82 'c' 송신 → 타겟이 GPIO4 트리거 발생 → 두 스코프 동시 캡처
        5. ★ 두 스코프 capture() 검사 (어느 한쪽이라도 실패면 freezing)
        6. 양쪽 트레이스 회수
        7. 결과 회수 (0x83 'r') → 페이로드 추출
        8. 6단계 분류 (FA_main 과 동일 로직)
        9. 결과 코드별 husky/lite 트레이스를 상한 내에서 보존
```

> ⏱ **실행 시간 안내**
> 예시 파라미터로도 5 × 4 × 13 × 5 ≈ **1300 시행** 입니다. 한 시행당 약 0.3 ~ 0.5초가 소요되므로 **약 7 ~ 11 분** 가량 걸립니다.
> 본격 연구 시에는 `i_offsets_pct`, `i_widths_pct` 의 격자를 더 촘촘히 (또는 반복 횟수를 늘려) 통계적 유의성을 확보하세요.

In [ ]:
# ══════════════════════════════════════════════════════════
#  메인 파라미터 탐색 루프 — Lite glitch + Husky wire-tap
# ══════════════════════════════════════════════════════════

REPS_PER_PARAM = 5  # 한 파라미터 조합당 반복 시행 횟수

for i_ext_offset in trange(i_ext_offsets[0], i_ext_offsets[-1] + 1,
                           desc='i_ext_offset', leave=False):

    for i_offset in i_offsets:
        for i_width in i_widths:

            # 유효성 검사
            if i_width == 0:
                continue
            if (i_offset + i_width) > PS:
                continue

            for _ in range(REPS_PER_PARAM):

                # 1. Lite 글리치 파라미터 설정
                lite_scope.glitch.ext_offset = i_ext_offset
                lite_scope.glitch.offset     = i_offset
                lite_scope.glitch.width      = i_width

                # 2. 타겟 초기화 + 데이터 주입
                reset_target_via_lite(lite_scope)
                my_fsr_cmd(target, 0x81, 'k', data_k)
                my_fsr_cmd(target, 0x81, 'p', data_p)
                my_fsr_cmd(target, 0x81, 'l', bytearray([MAX_DATA_LEN]))

                # 3. 두 스코프 동시 arm (Husky → Lite 순)
                target.flush()
                husky_scope.arm()
                lite_scope.arm()

                # 4. 연산 트리거 (펌웨어가 GPIO4 토글 → 두 스코프 동시 캡처)
                target.send_cmd(cmd=0x82, scmd=ord('c'), data=[])
                actual_ret = target.read_cmd(timeout=500)

                # 5. 캡처 검사 + 결과 코드 0 (freezing) 판정
                cap_husky = husky_scope.capture()
                cap_lite  = lite_scope.capture()
                if (actual_ret is None) or cap_lite:
                    # 타겟이 응답 없음 → freezing
                    code_now = 0
                    cglitch_result.append([code_now, i_ext_offset, i_width, i_offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    # Husky 가 캡처에 성공했다면 (cap_husky == 0) freezing 의 와이어태핑 단서로 보존
                    if (not cap_husky) and len(husky_by_code[code_now]) < MAX_TRACES_PER_CODE:
                        husky_by_code[code_now].append(np.array(husky_scope.get_last_trace()))
                        lite_by_code[code_now].append(np.array(lite_scope.get_last_trace()) if not cap_lite else None)
                        ctx_by_code[code_now].append((i_ext_offset, i_offset, i_width))
                    continue

                # 6. 두 스코프 트레이스 회수
                trace_husky = np.array(husky_scope.get_last_trace())
                trace_lite  = np.array(lite_scope.get_last_trace())

                # 7. 결과 페이로드 회수
                target.flush()
                target.send_cmd(cmd=0x83, scmd=ord('r'), data=[])
                actual_ret = target.read_cmd(timeout=500)
                if actual_ret is None:
                    code_now = 0
                    cglitch_result.append([code_now, i_ext_offset, i_width, i_offset])
                    cglitch_extra_data.append([expected_ret, actual_ret])
                    if len(husky_by_code[code_now]) < MAX_TRACES_PER_CODE:
                        husky_by_code[code_now].append(trace_husky)
                        lite_by_code[code_now].append(trace_lite)
                        ctx_by_code[code_now].append((i_ext_offset, i_offset, i_width))
                    continue
                actual_ret = actual_ret[3 : 3 + actual_ret[2]]

                # 8. 6단계 분류 (FA_main 로직 그대로)
                if expected_ret == actual_ret:
                    code_now = 1  # normal
                elif lite_scope.adc.trig_count < (lite_scope.adc.samples - 30):
                    code_now = 2  # for-loop skip (trig_count 가 크게 줄어듦)
                    n_diff = sum(g != r for g, r in zip(expected_ret, actual_ret))
                else:
                    n_diff = sum(g != r for g, r in zip(expected_ret, actual_ret))
                    if   n_diff == 1: code_now = 3
                    elif n_diff < 5:  code_now = 4
                    else:             code_now = 5

                cglitch_result.append([code_now, i_ext_offset, i_width, i_offset])
                cglitch_extra_data.append([expected_ret, actual_ret])

                # 9. 결과 코드별 husky/lite 파형 상한 내 보존
                if len(husky_by_code[code_now]) < MAX_TRACES_PER_CODE:
                    husky_by_code[code_now].append(trace_husky)
                    lite_by_code[code_now].append(trace_lite)
                    ctx_by_code[code_now].append((i_ext_offset, i_offset, i_width))

                # 흥미로운 결과는 진행 중에 출력
                if code_now in (2, 3):
                    LABEL = {2: 'loop skip', 3: '1-byte fault'}[code_now]
                    print(f'\n[코드{code_now}] {LABEL}  '
                          f'ext={i_ext_offset}  off={i_offset}  w={i_width}  '
                          f'ret_diff_bytes={sum(g!=r for g,r in zip(expected_ret, actual_ret))}')

print('\n[✓] 파라미터 탐색 완료')
print(f'  총 시행 수 : {len(cglitch_result)}')

---

# 📊 9단계 — 통계 분석으로 최적 글리치 파라미터 도출

> **이 단계의 목표**
> 결과 코드별 빈도와 *최빈* `(offset, width)` 를 산출합니다.
> 이는 `FA_main.ipynb` 4단계와 동일한 분석이며, 다음 10단계 시각화의 기반이 됩니다.

---

In [ ]:
# 결과 리스트 → numpy 배열
cglitch_result_arr = np.array(cglitch_result, dtype=np.float64)

print(f'전체 시행 횟수 : {len(cglitch_result_arr)}')
print(f'배열 shape    : {cglitch_result_arr.shape}  (rows, [code, ext_offset, width, offset])')

In [ ]:
# 결과 코드별 빈도 + 최빈 (offset, width)
LABELS = {
    0: 'freezing',
    1: 'normal',
    2: 'for-loop skip',
    3: 'one faulty byte',
    4: 'few faulty bytes (<5)',
    5: 'etc (>=5 faults)',
}

print('=' * 64)
print(f'{"코드":>5} {"분류":<24} {"건수":>8} {"best offset":>14} {"best width":>10}')
print('=' * 64)

for code_, label in LABELS.items():
    mask = cglitch_result_arr[:, 0] == code_
    cnt  = int(mask.sum())

    if cnt == 0:
        print(f'{code_:>5} {label:<24} {cnt:>8} {"—":>14} {"—":>10}')
        continue

    best_offset = sp.stats.mode(cglitch_result_arr[mask, 3], keepdims=False).mode
    best_width  = sp.stats.mode(cglitch_result_arr[mask, 2], keepdims=False).mode
    print(f'{code_:>5} {label:<24} {cnt:>8} {int(best_offset):>14} {int(best_width):>10}')

print('=' * 64)

> 💡 **결과 해석 가이드** (FA_main 과 동일)
>
> - **코드 2 (for-loop skip)** 의 `best (offset, width)` → **인증 우회 공격용** 후보 파라미터
> - **코드 3 (1-byte fault)** 의 `best (offset, width)` → **DFA 키 복원용** 후보 파라미터
> - 코드 0(freezing) 비율이 높으면 → 글리치가 너무 강함 (`i_width` 를 줄일 것)
> - 코드 1(normal) 비율이 절대 다수면 → 글리치가 너무 약함 (`i_width` 를 늘릴 것)
>
> ★ **본 노트북의 추가 기능**
> 이번에는 Husky 와이어태핑 파형으로 *코드 2/3 발생 시의 클럭·전력 패턴* 도 확인할 수 있습니다 (다음 10단계).

---

# 🎨 10단계 — 시각화: 파라미터 분포 + 와이어태핑 파형 비교

> **이 단계의 목표**
> 본 노트북의 핵심 결과 3가지를 인터랙티브 그래프로 제시합니다.
>
> 1. **(offset, width) 평면 분포** — `FA_main` 의 5단계와 동일 (단, Lite 의 `PS` 기준)
> 2. ★ **결과 코드별 Husky 와이어태핑 파형 비교** — 본 노트북 고유 산출물
> 3. ★ **Lite 자체 측정 vs Husky 와이어태핑 비교** — 측정 경로 신뢰성 검증

---

### 10.1 (offset, width) 평면 산점도

같은 결과 코드가 **공간적으로 군집** 을 이루는 영역이 곧 **재현성 높은 글리치 파라미터 영역** 입니다.

In [ ]:
# 색상 팔레트 (코드 0~5)
COLOR_MAP = {
    0: '#888888',   # freezing      — gray
    1: '#1f77b4',   # normal        — blue
    2: '#2ca02c',   # loop skip     — green ✅
    3: '#d62728',   # 1-byte fault  — red ✅
    4: '#ff7f0e',   # few faults    — orange
    5: '#9467bd',   # etc           — purple
}

p_map = figure(
    width=900, height=500,
    title='Glitch Parameter Map — (offset, width) plane  [Lite PS-relative]',
    x_axis_label='i_offset (1-clock 내 시작 위상, Lite phase_shift_steps 기준)',
    y_axis_label='i_width  (글리치 펄스 폭, Lite phase_shift_steps 기준)',
    tools='pan,wheel_zoom,box_zoom,reset,save',
    active_scroll='wheel_zoom',
    background_fill_color='#fafafa',
)

for code_, label in LABELS.items():
    mask = cglitch_result_arr[:, 0] == code_
    if not mask.any():
        continue
    src = ColumnDataSource(data=dict(
        x = cglitch_result_arr[mask, 3],   # offset
        y = cglitch_result_arr[mask, 2],   # width
        ext = cglitch_result_arr[mask, 1], # ext_offset
        label = [label] * int(mask.sum()),
    ))
    p_map.scatter('x', 'y', source=src,
                  size=8, alpha=0.55,
                  color=COLOR_MAP[code_],
                  legend_label=f'[{code_}] {label} (n={int(mask.sum())})')

p_map.title.text_font_size = '13pt'
p_map.title.align = 'center'
p_map.grid.grid_line_alpha = 0.3
p_map.outline_line_color = None
p_map.legend.location = 'top_right'
p_map.legend.click_policy = 'hide'
p_map.legend.label_text_font_size = '9pt'
p_map.legend.background_fill_alpha = 0.85
p_map.add_tools(HoverTool(tooltips=[
    ('result', '@label'),
    ('i_offset', '@x{0}'),
    ('i_width',  '@y{0}'),
    ('i_ext_offset', '@ext{0}'),
]))

show(p_map)

### 10.2 결과 코드별 Husky 와이어태핑 파형 비교 (★ 본 노트북의 핵심 산출물)

각 결과 코드별로 보존된 Husky 와이어태핑 파형을 **겹쳐서** 그립니다.
주된 비교 포인트:

| 비교 쌍 | 의미 |
|:----:|:----|
| 코드 1 vs 코드 2 | *언제* 글리치가 for-loop 종료 조건을 무너뜨리는가? — 트리거 후 N clock 부근에서 파형이 *짧아짐* |
| 코드 1 vs 코드 3 | *언제* 글리치가 단일 XOR 연산을 변조하는가? — 특정 클럭 구간에서 *특이 피크* |
| 코드 0(freezing) | 글리치가 PC 를 망가뜨려 *비정상 영역으로 점프* — 파형이 무작위적으로 발산 |

> 🔬 **핵심 통찰**
> 같은 결과 코드 내에서도 파형 군집이 **시간축 상으로 일관** 하다면, 이는 글리치가 *항상 같은 명령어* 에 작용했음을 의미합니다.
> 반대로 군집이 흩어진다면 *우연한 성공* 일 가능성이 큽니다 → 더 많은 반복 시행으로 검증 필요.

In [ ]:
# 결과 코드별 husky 파형 겹쳐 그리기
# (보존된 파형이 있는 코드만 패널로 출력)

panels = []
for code_, label in LABELS.items():
    traces = husky_by_code.get(code_, [])
    if len(traces) == 0:
        continue

    arr = np.array(traces)
    n_show = min(len(traces), MAX_TRACES_PER_CODE)
    sample_axis_clk = np.arange(arr.shape[1]) / 4.0   # 4× oversample → clock units

    p = figure(
        width=900, height=240,
        title=f'Husky wire-tap — Code {code_} : {label}  (n={n_show})',
        x_axis_label='Clock Cycle from Trigger',
        y_axis_label='V',
        tools='pan,wheel_zoom,box_zoom,reset,save',
        active_scroll='wheel_zoom',
        background_fill_color='#fafafa',
    )
    palette = Category10[10]
    for i in range(n_show):
        p.line(sample_axis_clk, arr[i],
               line_width=1.0,
               line_color=COLOR_MAP[code_],
               line_alpha=0.40)
    # 평균 파형도 강조 표시
    mean_trace = arr[:n_show].mean(axis=0)
    p.line(sample_axis_clk, mean_trace,
           line_width=2.0, line_color='#1a1a1a', line_alpha=0.95,
           legend_label='mean trace')
    p.title.text_font_size = '11pt'
    p.grid.grid_line_alpha = 0.3
    p.outline_line_color = None
    p.legend.location = 'top_right'
    p.legend.label_text_font_size = '9pt'
    panels.append(p)

if panels:
    show(column(*panels))
else:
    print('보존된 husky 파형이 없습니다. 8단계가 정상 실행되었는지 확인하세요.')

### 10.3 동일 시행에서의 Lite 자체 측정 vs Husky 와이어태핑 비교

가장 흥미로운 결과 코드 (코드 2 또는 3) 에서 **첫 번째 보존 시행** 을 골라 Lite·Husky 양쪽 파형을 비교합니다.
두 파형의 거시 패턴이 일치한다면 → 와이어태핑 경로가 신뢰할 만함을 시각적으로 입증.

> 🔬 **이 비교의 연구적 가치**
> 단일 장치 FIA 에서는 *내 측정값만* 의지할 수밖에 없습니다.
> 다중 장치 결합에서는 **두 시점의 측정으로 상호 검증** 이 가능하므로, 단일 장치에서는 발견하기 어려운 측정 아티팩트(예: 자체 글리치가 자체 ADC 에 미치는 영향) 를 가려낼 수 있습니다.

In [ ]:
# 코드 2 또는 3 중 보존된 첫 번째 시행을 비교 대상으로 선택
candidate_code = None
for c_ in (2, 3, 4, 5, 0):
    if len(husky_by_code[c_]) > 0:
        candidate_code = c_
        break

if candidate_code is None:
    print('비교에 사용할 보존 시행이 없습니다.')
else:
    husky_t = husky_by_code[candidate_code][0]
    lite_t  = lite_by_code [candidate_code][0]
    ctx     = ctx_by_code  [candidate_code][0]
    label   = LABELS[candidate_code]

    print(f'비교 대상: Code {candidate_code} ({label})')
    print(f'  ext_offset = {ctx[0]}  /  i_offset = {ctx[1]}  /  i_width = {ctx[2]}')

    # Lite (1 sample = 1 clock)
    p_lite = figure(
        width=900, height=260,
        title=f'Lite self-measure (1 sample = 1 clock) — Code {candidate_code} : {label}',
        x_axis_label='Sample (= Clock from Trigger)',
        y_axis_label='V',
        tools='pan,wheel_zoom,box_zoom,reset,save',
        active_scroll='wheel_zoom',
        background_fill_color='#fafafa',
    )
    p_lite.line(np.arange(len(lite_t)), lite_t, line_width=1.2, line_color='#2E86AB')
    p_lite.grid.grid_line_alpha = 0.3
    p_lite.outline_line_color = None
    # 글리치 발생 시점에 수직선 표시
    p_lite.add_layout(Span(location=ctx[0], dimension='height',
                           line_color='#d62728', line_dash='dashed', line_width=1.5))

    # Husky (4 samples / clock)
    husky_x_clk = np.arange(len(husky_t)) / 4.0
    p_husky = figure(
        width=900, height=260,
        title=f'Husky wire-tap (4× oversample) — Code {candidate_code} : {label}',
        x_axis_label='Clock Cycle from Trigger',
        y_axis_label='V',
        tools='pan,wheel_zoom,box_zoom,reset,save',
        active_scroll='wheel_zoom',
        background_fill_color='#fafafa',
    )
    p_husky.line(husky_x_clk, husky_t, line_width=1.0, line_color='#A23B72', line_alpha=0.9)
    p_husky.grid.grid_line_alpha = 0.3
    p_husky.outline_line_color = None
    p_husky.add_layout(Span(location=ctx[0], dimension='height',
                            line_color='#d62728', line_dash='dashed', line_width=1.5))

    show(column(p_lite, p_husky))

> 💡 **두 파형 비교에서 점검할 사항**
>
> - **빨간 점선 = 글리치 발생 시점 (`ext_offset`)** : 두 그래프에서 동일한 x 좌표에 그려집니다. 이 시점 *직전·직후* 의 파형 변화를 관찰하세요.
> - **클럭 라인 와이어태핑** (Husky 가 AUX MCX 로 보는 신호) 은 글리치 펄스가 **사각파에 짧은 추가 펄스로** 나타납니다.
> - **거시 진폭/시간축** 이 일치한다면 와이어태핑 경로가 정상 측정에 준하는 신뢰도임을 시사.

---

# 🔚 마무리 — 다중 장치 자원 해제

> **이 단계의 목표**
> 노트북 종료 전에 **target → 두 scope** 순서로 명시적 해제해 다음 세션의 USB·UART 충돌을 방지합니다.

---

해제 순서를 반드시 지켜야 하는 이유 (Wiretapping4SCA 와 동일):

1. **`target.dis()`** — Lite 가 점유 중인 UART 채널 해제
2. **각 scope.dis()** — Husky, Lite 순으로 USB 핸들 해제

> ⚠️ **타겟을 먼저 해제하지 않으면** Lite 의 UART 핀이 점유된 채로 남아 차회 실행 시 `target` 재생성이 실패할 수 있습니다.

In [ ]:
def disconnect_all_devices(scopes: dict) -> None:
    # 타겟 객체 먼저 닫기 (Lite 의 UART 점유 해제)
    try:
        target.dis()
        print("  [✓] 타겟 보드 (SimpleSerial2) 연결 해제 완료")
    except Exception as e:
        print(f"  [✗] 타겟 보드 연결 해제 실패  └─ {e}")

    print("\n장치 연결 해제 중...")
    for name, scope in scopes.items():
        try:
            scope.dis()
            print(f"  [✓] {name} 연결 해제 완료")
        except Exception as e:
            print(f"  [✗] {name} 연결 해제 실패\n      └─ {e}")
    scopes.clear()

# 파형 수집 및 데이터 저장 완료 후 반드시 자원 반환
disconnect_all_devices(scopes)

---

## 📝 본 노트북 요약

| 단계 | 핵심 함수 / 명령 | 결과 |
|:----:|:---|:---|
| 1 | `cw.list_devices()` + `cw.scope(sn=...)`             | 다중 장치 동시 연결 (Lite + Husky) |
| 2 | `cw.target(lite_scope, SimpleSerial2)`              | Lite ↔ 타겟 통신 채널 확립 |
| 3 | `make` + `cw.program_target(lite_scope, ...)`       | Lite 가 프로그래머로 동작 |
| 4 | `my_fsr_cmd()` + Golden Model 비교                   | 통신·연산 정상성 검증 |
| 5 | `lite_scope.cglitch_setup()` + `adc_mul=1`          | **Lite 에서 1 sample = 1 clock 정렬 + 글리치 모듈 활성** |
| 6 | `extclk_aux_io` + `freq_ctr` + `adc_mul=4`          | Husky 가 외부 클럭(글리치 합성) 에 PLL 잠금, 4× 와이어태핑 |
| 7 | `arm_timing='no_glitch'` + 양쪽 동시 캡처             | 베이스라인 `expected_ret` + 두 시점 파형 |
| 8 | 두 스코프 동시 arm + `lite.glitch.{ext_offset,offset,width}` | 결과 코드별 husky/lite 파형 보존 |
| 9 | `np.array` + `sp.stats.mode`                        | 결과 코드별 최빈 `(offset, width)` |
| 10 | Bokeh: 산점도 + 코드별 평균 파형 + Lite/Husky 비교    | 인터랙티브 분포·파형 시각화 |

### ✅ 본 노트북에서 익혀야 할 핵심 개념

1. **결합 위협 모델** — 한 공격자가 능동 FIA + 수동 SCA 능력을 동시에 보유한 시나리오의 실험적 재현
2. **두 ADC 의 비대칭 운용** — Lite `adc_mul=1` (FIA 시점 정밀) + Husky `adc_mul=4` (글리치 형태 관찰)
3. **글리치된 클럭 라인의 외부 와이어태핑** — Husky 의 AUX MCX 가 *Lite 가 출력한 글리치 합성 클럭* 을 직접 측정
4. **두 스코프 동시 arm 패턴** — 단일 트리거로 두 파형을 시간축 정렬 캡처
5. **결과 코드별 와이어태핑 파형 보존** — loop-skip / 1-byte fault 등 각 fault category 의 *물리적 신호 단서* 추출

### 🔬 후속 연구 방향 제안

- **결합 시나리오로 SIFA(Statistical Ineffective Fault Attack) 구현** — 코드 1(normal) 의 husky 파형 분포로 키 종속 누설을 분석
- **DFA + SCA 융합** — 코드 3(1-byte fault) 의 husky 파형에서 키 후보별 가설 검증
- **글리치 파형의 정량 분석** — Husky 클럭 라인 측정으로 *의도한 width* 대비 *실측 width* 의 분포 분석
- **다중 글리치 결합** — Lite 의 글리치를 다중 펄스로 확장하면서 husky 로 펄스 간 간격을 검증

---
*결합 위협 모델 — Husky 와이어태핑 + Lite 오류주입 노트북 끝*